# UNet Architecture Visualization

This notebook mirrors `visualization.py` and supports two workflows:
- **Experiment comparison**: compare UNet / UNet++ / UNet 3+ inside one experiment target
- **Depth comparison**: compare the same architecture across different depths

All figures are saved under `visualization/` using the same folder rules as the CLI script.


In [ ]:
import matplotlib.pyplot as plt
from visualization import (
    load_experiments,
    _latest_per_model,
    select_latest_per_depth,
    resolve_experiment_target,
    describe_target,
    build_output_dir_for_target,
    build_depth_comparison_output_dir,
    build_depth_style_map,
    list_available_model_names,
    normalize_model_name,
    plot_training_curves,
    plot_eigenvalue_spectrum,
    plot_theoretical_bars,
    plot_correlation_scatter,
    plot_empirical_summary,
    print_ranking_table,
    save_ranking_table,
)
%matplotlib inline


## 0. Configuration


In [ ]:
VIS_MODE = "experiment"  # "experiment" or "depth_comparison"
EXPERIMENTS_DIR = "experiments"
EXPERIMENT_TARGET = "depth=4"  # used when VIS_MODE == "experiment"
DEPTH_COMPARE_MODEL = "UNet3Plus"  # used when VIS_MODE == "depth_comparison"
FIGURES_ROOT = "visualization"

if VIS_MODE == "experiment":
    TARGET_DIR = resolve_experiment_target(EXPERIMENTS_DIR, EXPERIMENT_TARGET)
    OUTPUT_DIR = build_output_dir_for_target(FIGURES_ROOT, TARGET_DIR, EXPERIMENTS_DIR)
    print(f"Selected experiment target: {describe_target(TARGET_DIR, EXPERIMENTS_DIR)}")
    print(f"Source directory: {TARGET_DIR}")
elif VIS_MODE == "depth_comparison":
    DEPTH_COMPARE_MODEL = normalize_model_name(DEPTH_COMPARE_MODEL)
    TARGET_DIR = None
    OUTPUT_DIR = build_depth_comparison_output_dir(FIGURES_ROOT, DEPTH_COMPARE_MODEL)
    print(f"Selected depth-comparison model: {DEPTH_COMPARE_MODEL}")
else:
    raise ValueError(f"Unsupported VIS_MODE: {VIS_MODE}")

print(f"Output directory: {OUTPUT_DIR}")


## 1. Load Selected Data


In [ ]:
# The notebook reuses visualization.py so the CLI and notebook stay in sync.


In [ ]:
all_records = load_experiments(EXPERIMENTS_DIR, experiments_root=EXPERIMENTS_DIR)
if not all_records:
    raise FileNotFoundError(f"No experiment results found under: {EXPERIMENTS_DIR}")

available_models = list_available_model_names(all_records)
available_models_text = ", ".join(available_models)
print(f"Available model families: {available_models_text}")

if VIS_MODE == "experiment":
    scoped_records = load_experiments(TARGET_DIR, experiments_root=EXPERIMENTS_DIR)
    if not scoped_records:
        raise FileNotFoundError(f"No experiment results found under: {TARGET_DIR}")
    records = _latest_per_model(scoped_records)
    STYLE_MAP = None
    loaded_items_text = ", ".join(records.keys())
    print(f"Loaded {len(records)} model(s): {loaded_items_text}")
else:
    model_records = [record for record in all_records if record["model_name"] == DEPTH_COMPARE_MODEL]
    if not model_records:
        raise ValueError(f"No records found for model: {DEPTH_COMPARE_MODEL}")
    records = select_latest_per_depth(model_records)
    if not records:
        raise ValueError(f"No depth-tagged records found for model: {DEPTH_COMPARE_MODEL}")
    STYLE_MAP = build_depth_style_map(records.keys())
    loaded_items_text = ", ".join(records.keys())
    print(f"Loaded depths: {loaded_items_text}")

print(f"Figures will be saved to: {OUTPUT_DIR}")


## 2. Removed DAG Topology

`dag_topology.pdf` generation has been removed from the project.


In [ ]:
print("[skip] DAG topology generation has been removed.")


## 3. Training Curves


In [ ]:
plot_training_curves(records, OUTPUT_DIR, style_map=STYLE_MAP, show=True)


## 4. Eigenvalue Spectrum


In [ ]:
plot_eigenvalue_spectrum(records, OUTPUT_DIR, style_map=STYLE_MAP, show=True)


## 5. Theoretical Metrics Comparison


In [ ]:
plot_theoretical_bars(records, OUTPUT_DIR, style_map=STYLE_MAP, show=True)


## 6. Correlation Analysis


In [ ]:
plot_correlation_scatter(records, OUTPUT_DIR, style_map=STYLE_MAP, show=True)


## 7. Empirical Summary


In [ ]:
plot_empirical_summary(records, OUTPUT_DIR, style_map=STYLE_MAP, show=True)


## 8. Multi-Objective Ranking


In [ ]:
print_ranking_table(records, style_map=STYLE_MAP)


In [ ]:
save_ranking_table(records, OUTPUT_DIR, style_map=STYLE_MAP)
